# CIS 531/731: Lab 4 - Spark Structured Streaming
**Phase II | Module 4: Data Analytics Pipelines & Streaming**

### Lab Objectives
In this lab, we operationalize the concepts from *The Dataflow Model*. You will:
1. **Initialize** a PySpark structured streaming context.
2. **Ingest** a simulated live continuous data feed.
3. **Configure** event-time watermarks to bound late-arriving data.
4. **Execute** a windowed aggregation query to compute rolling metrics.
5. **Route** the processed micro-batches to a storage sink.

---
### Prerequisites
Run the cell below to initialize your Spark session. We are running locally in this notebook to observe the stream mechanics before deploying to a cluster.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, rand, expr

# 1. Initialize the PySpark Session
spark = SparkSession.builder \
    .appName("CIS531_731_StructuredStreaming_Lab") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session Initialized!")

### Step 1: Ingesting a Simulated Data Stream
To guarantee execution without external dependencies (like Kafka), we use Spark's native `rate` stream. It automatically generates rows containing a `timestamp` and a monotonically increasing `value` at a specified rows-per-second rate.

In [ ]:
# 2. Connect to a simulated continuous data feed
# Emits 5 rows per second
raw_stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Simulate a 'category' attribute for our stream data using random assignment
enriched_stream_df = raw_stream_df.withColumn(
    "event_category", 
    expr("CASE WHEN rand() > 0.5 THEN 'Type_A' ELSE 'Type_B' END")
)

enriched_stream_df.printSchema()

### Step 2: Watermarking & Windowed Aggregations
We group the stream into **10-second tumbling windows**. 
To handle late data without keeping unbounded state in memory, we apply a **Watermark** of 15 seconds. Any data arriving more than 15 seconds late relative to the maximum observed event time will be dropped.

In [ ]:
# 3 & 4. Configure watermarks and windowed aggregation
windowed_aggregations = enriched_stream_df \
    .withWatermark("timestamp", "15 seconds") \
    .groupBy(
        window(col("timestamp"), "10 seconds"),
        col("event_category")
    ) \
    .count()

print("Aggregation query constructed. Ready for sink routing.")

### Step 3: Sinking to Storage
In a production Data Lake, you would route the `writeStream` to a Parquet or Delta Lake format. For this interactive notebook, we will route it to an in-memory table so we can query the micro-batches using standard SQL.

In [ ]:
import time

# 5. Route output to a memory sink for interactive querying
streaming_query = windowed_aggregations.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("live_metric_aggregations") \
    .start()

print("Streaming Query Started! Polling the memory sink for 30 seconds...")

# Poll the memory sink table every 5 seconds to observe the continuous analytics
for i in range(6):
    time.sleep(5)
    print(f"\n--- Micro-batch Update {i+1} ---")
    spark.sql("SELECT * FROM live_metric_aggregations ORDER BY window.start DESC").show(truncate=False)


### Step 4: Graceful Shutdown
Always explicitly halt your streaming queries to free up cluster resources.

In [ ]:
# Stop the streaming query to release resources
streaming_query.stop()
print("Streaming query successfully halted.")